# llama model checkpointing - cpu-only cluster version

comprehensive comparison of pytorch vs tensorstore approaches
- all methods save complete model (all 237 parameters)
- cpu-only execution
- no inference required

In [1]:
# import required libraries
import torch
import time
import os
from transformers import LlamaForCausalLM
import gc
import tensorstore as ts
import numpy as np
import matplotlib.pyplot as plt
import json

In [2]:
# cpu only execution
device = torch.device('cpu')
print(f"using device: {device}")
print(f"cpu count: {os.cpu_count()}")

using device: cpu
cpu count: 8


In [3]:
# create output directory
os.makedirs('saved_models', exist_ok=True)
print("created saved_models directory")

created saved_models directory


In [4]:
# load openllama-3b model
model_name = "openlm-research/open_llama_3b"
print(f"loading model: {model_name}")

# load pre-trained llama model from huggingface hub
model = LlamaForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # load in float16 to reduce memory usage
    low_cpu_mem_usage=True  # optimize loading to use less cpu memory
)

# move to cpu explicitly
# explicitly move model to cpu (ensures all tensors are on cpu)
model = model.to('cpu')

print(f"model loaded successfully")
print(f"total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}m")
print(f"parameter count: {len(list(model.named_parameters()))}")

`torch_dtype` is deprecated! Use `dtype` instead!


loading model: openlm-research/open_llama_3b
model loaded successfully
total parameters: 3426.5m
parameter count: 237


## phase 1: pytorch approach

In [5]:
# pytorch save - all parameters
pytorch_save_path = "saved_models/openllama_3b_pytorch.pth"

print("=== phase 1: pytorch saving ===")
start_time = time.time()

# save all model parameters using pytorch's native serialization format
torch.save(model.state_dict(), pytorch_save_path)

pytorch_save_time = time.time() - start_time
pytorch_file_size = os.path.getsize(pytorch_save_path) / (1024**3)

print(f"pytorch save completed in {pytorch_save_time*1000:.1f} ms")
print(f"file size: {pytorch_file_size:.2f} gb")
print(f"saved to: {pytorch_save_path}")

=== phase 1: pytorch saving ===
pytorch save completed in 8061.1 ms
file size: 6.38 gb
saved to: saved_models/openllama_3b_pytorch.pth


In [6]:
# pytorch load
print("\n=== phase 1: pytorch loading ===")
start_time = time.time()

# load model parameters from pytorch checkpoint
# map_location='cpu': load tensors to cpu memory (not gpu)
# weights_only=True: security feature to prevent arbitrary code execution
state_dict = torch.load(pytorch_save_path, map_location='cpu', weights_only=True)

pytorch_load_time = time.time() - start_time

print(f"pytorch load completed in {pytorch_load_time*1000:.1f} ms")
print(f"loaded {len(state_dict)} parameters successfully")

del state_dict
gc.collect()


=== phase 1: pytorch loading ===
pytorch load completed in 6397.0 ms
loaded 237 parameters successfully


56

## phase 2: basic tensorstore approach

In [7]:
# dynamic chunking helper function (used by phase 2 and phase 3)
def calculate_chunk_shape(shape, target_elements):
    # calculate optimal chunk shape to target a specific number of elements
    # used to create ~64mib chunks for efficient i/o
    
    # ensure minimum chunk size of 1 element
    if target_elements < 1:
        target_elements = 1
    if not shape:
        return [1]
    
    chunk_shape = list(shape)
    
    # iteratively halve the largest dimension until chunk size fits target
    while np.prod(chunk_shape) > target_elements and max(chunk_shape) > 1:
        max_idx = chunk_shape.index(max(chunk_shape))
        chunk_shape[max_idx] = max(1, chunk_shape[max_idx] // 2)
    
    return chunk_shape

print("dynamic chunking helper function loaded")

dynamic chunking helper function loaded


In [8]:
# reusable tensorstore functions
def save_with_tensorstore(model_state, save_dir, method_name, 
                         use_compression=False, use_concurrency=False, 
                         chunk_size_mb=64, custom_chunk_fn=None):
    """
    save model using tensorstore with configurable options
    
    args:
        model_state: model.state_dict()
        save_dir: directory to save files
        method_name: name for this method (for logging)
        use_compression: whether to use gzip compression
        use_concurrency: whether to use high concurrency context
        chunk_size_mb: target chunk size in megabytes
        custom_chunk_fn: custom function to calculate chunk shape
    """
    import time
    import os
    import json
    import tensorstore as ts
    
    os.makedirs(save_dir, exist_ok=True)
    print(f"\n=== {method_name}: saving ===")
    start_time = time.time()
    
    # setup context if using concurrency
    context = ts.Context({'file_io_concurrency': {'limit': 128}}) if use_concurrency else None
    
    # calculate chunk size in bytes
    chunk_size_bytes = chunk_size_mb * 1024 * 1024
    
    saved_count = 0
    for param_name, param_tensor in model_state.items():
        try:
            # convert to numpy
            param_np = param_tensor.detach().cpu().half().numpy()
            
            # create safe filename
            safe_name = param_name.replace('.', '_').replace('/', '_')
            
            # calculate chunk shape
            if custom_chunk_fn:
                chunk_shape = custom_chunk_fn(param_np.shape, param_np.dtype)
            else:
                target_elements = chunk_size_bytes // param_np.dtype.itemsize
                chunk_shape = calculate_chunk_shape(list(param_np.shape), target_elements)
            
            # build spec
            spec = {
                'driver': 'zarr',
                'kvstore': {
                    'driver': 'file',
                    'path': f"{save_dir}{safe_name}.zarr"
                },
                'metadata': {
                    'shape': list(param_np.shape),
                    'dtype': '<f2',
                    'chunks': chunk_shape
                }
            }
            
            # add compression if requested
            if use_compression:
                spec['metadata']['compressor'] = {'id': 'gzip', 'level': 1}
            
            # open and write
            if context:
                store = ts.open(spec, create=True, delete_existing=True, context=context).result()
            else:
                store = ts.open(spec, create=True, delete_existing=True).result()
            
            store.write(param_np).result()
            saved_count += 1
            
        except Exception as e:
            print(f"error saving {param_name}: {e}")
            continue
    
    # save metadata
    metadata = {
        'param_names': list(model_state.keys()),
        'total_params': len(model_state),
        'method': method_name,
        'chunk_size_mb': chunk_size_mb,
        'compression': use_compression,
        'concurrency': use_concurrency
    }
    with open(f"{save_dir}metadata.json", 'w') as f:
        json.dump(metadata, f)
    
    save_time = time.time() - start_time
    
    # calculate size
    total_size = 0
    for root, dirs, files in os.walk(save_dir):
        for file in files:
            total_size += os.path.getsize(os.path.join(root, file))
    file_size_gb = total_size / (1024**3)
    
    print(f"{method_name} save completed in {save_time*1000:.1f} ms")
    print(f"saved {saved_count} parameters")
    print(f"total size: {file_size_gb:.2f} gb")
    
    return save_time, file_size_gb


def load_with_tensorstore(save_dir, method_name):
    """
    load model from tensorstore
    
    args:
        save_dir: directory where files are saved
        method_name: name for this method (for logging)
    """
    import time
    import os
    import json
    import tensorstore as ts
    import torch
    
    print(f"\n=== {method_name}: loading ===")
    start_time = time.time()
    
    # load metadata
    with open(f"{save_dir}metadata.json", 'r') as f:
        metadata = json.load(f)
    
    loaded_state = {}
    loaded_count = 0
    
    for param_name in metadata['param_names']:
        try:
            safe_name = param_name.replace('.', '_').replace('/', '_')
            zarr_path = f"{save_dir}{safe_name}.zarr"
            
            if os.path.exists(zarr_path):
                spec = {
                    'driver': 'zarr',
                    'kvstore': {
                        'driver': 'file',
                        'path': zarr_path
                    }
                }
                
                store = ts.open(spec).result()
                param_np = store.read().result()
                loaded_state[param_name] = torch.from_numpy(param_np.copy())
                loaded_count += 1
                
        except Exception as e:
            print(f"error loading {param_name}: {e}")
            continue
    
    load_time = time.time() - start_time
    
    print(f"{method_name} load completed in {load_time*1000:.1f} ms")
    print(f"loaded {loaded_count} parameters successfully")
    
    del loaded_state
    import gc
    gc.collect()
    
    return load_time

print("reusable tensorstore functions loaded")

reusable tensorstore functions loaded


In [9]:
# async tensorstore functions for t5x (phase 3)
import asyncio

async def save_with_tensorstore_async(model_state, save_dir, method_name,
                                     use_compression=True, use_concurrency=True,
                                     chunk_size_mb=64):
    """
    async save model using tensorstore (t5x approach)
    
    args:
        model_state: model.state_dict()
        save_dir: directory to save files
        method_name: name for this method (for logging)
        use_compression: whether to use gzip compression
        use_concurrency: whether to use high concurrency context
        chunk_size_mb: target chunk size in megabytes
    """
    import time
    import os
    import json
    import tensorstore as ts
    
    # atomic write: use temporary directory
    timestamp = int(time.time())
    tmp_dir = f"{save_dir}.tmp-{timestamp}"
    os.makedirs(tmp_dir, exist_ok=True)
    
    print(f"\n=== {method_name}: saving ===")
    start_time = time.time()
    
    # setup context if using concurrency
    context = ts.Context({'file_io_concurrency': {'limit': 128}}) if use_concurrency else None
    
    # calculate chunk size in bytes
    chunk_size_bytes = chunk_size_mb * 1024 * 1024
    
    async def write_param(param_name, param_tensor):
        """async write single parameter"""
        try:
            # convert to numpy
            param_np = param_tensor.detach().cpu().half().numpy()
            
            # create safe filename (relative path for portability)
            safe_name = param_name.replace('.', '_').replace('/', '_')
            
            # calculate chunk shape
            target_elements = chunk_size_bytes // param_np.dtype.itemsize
            chunk_shape = calculate_chunk_shape(list(param_np.shape), target_elements)
            
            # build spec with relative path
            spec = {
                'driver': 'zarr',
                'kvstore': {
                    'driver': 'file',
                    'path': f"{tmp_dir}{safe_name}.zarr"  # write to tmp first
                },
                'metadata': {
                    'shape': list(param_np.shape),
                    'dtype': '<f2',
                    'chunks': chunk_shape
                }
            }
            
            # add compression if requested
            if use_compression:
                spec['metadata']['compressor'] = {'id': 'gzip', 'level': 1}
            
            # async open and write
            if context:
                store = await ts.open(spec, create=True, delete_existing=True, context=context)
            else:
                store = await ts.open(spec, create=True, delete_existing=True)
            
            await store.write(param_np)
            return param_name, True
            
        except Exception as e:
            print(f"error saving {param_name}: {e}")
            return param_name, False
    
    # create async tasks for all parameters
    tasks = [write_param(name, tensor) for name, tensor in model_state.items()]
    
    # execute all writes concurrently
    results = await asyncio.gather(*tasks)
    saved_count = sum(1 for _, success in results if success)
    
    # save metadata with version info
    metadata = {
        'version': 3,  # t5x version
        'param_names': list(model_state.keys()),
        'total_params': len(model_state),
        'method': method_name,
        'chunk_size_mb': chunk_size_mb,
        'compression': use_compression,
        'concurrency': use_concurrency,
        'save_timestamp': time.time()
    }
    with open(f"{tmp_dir}metadata.json", 'w') as f:
        json.dump(metadata, f, indent=2)
    
    save_time = time.time() - start_time
    
    # calculate size
    total_size = 0
    for root, dirs, files in os.walk(tmp_dir):
        for file in files:
            total_size += os.path.getsize(os.path.join(root, file))
    file_size_gb = total_size / (1024**3)
    
    # atomic rename: move tmp to final location
    if os.path.exists(save_dir):
        import shutil
        shutil.rmtree(save_dir)
    os.rename(tmp_dir, save_dir)
    
    print(f"{method_name} save completed in {save_time*1000:.1f} ms")
    print(f"saved {saved_count} parameters")
    print(f"total size: {file_size_gb:.2f} gb")
    
    return save_time, file_size_gb


async def load_with_tensorstore_async(save_dir, method_name):
    """
    async load model from tensorstore (t5x approach)
    
    args:
        save_dir: directory where files are saved
        method_name: name for this method (for logging)
    """
    import time
    import os
    import json
    import tensorstore as ts
    import torch
    
    print(f"\n=== {method_name}: loading ===")
    start_time = time.time()
    
    # load metadata
    with open(f"{save_dir}metadata.json", 'r') as f:
        metadata = json.load(f)
    
    async def load_param(param_name):
        """async load single parameter"""
        try:
            safe_name = param_name.replace('.', '_').replace('/', '_')
            zarr_path = f"{save_dir}{safe_name}.zarr"
            
            if os.path.exists(zarr_path):
                # open spec without strict metadata requirements (compatibility)
                spec = {
                    'driver': 'zarr',
                    'kvstore': {
                        'driver': 'file',
                        'path': zarr_path
                    }
                }
                
                store = await ts.open(spec)
                param_np = await store.read()
                return param_name, torch.from_numpy(param_np.copy())
            return param_name, None
                
        except Exception as e:
            print(f"error loading {param_name}: {e}")
            return param_name, None
    
    # create async tasks for all parameters
    tasks = [load_param(name) for name in metadata['param_names']]
    
    # execute all loads concurrently
    results = await asyncio.gather(*tasks)
    
    # build state dict
    loaded_state = {}
    loaded_count = 0
    for param_name, param_tensor in results:
        if param_tensor is not None:
            loaded_state[param_name] = param_tensor
            loaded_count += 1
    
    load_time = time.time() - start_time
    
    print(f"{method_name} load completed in {load_time*1000:.1f} ms")
    print(f"loaded {loaded_count} parameters successfully")
    
    del loaded_state
    import gc
    gc.collect()
    
    return load_time


def run_async_save(model_state, save_dir, method_name, **kwargs):
    """wrapper to run async save in sync context"""
    return asyncio.run(save_with_tensorstore_async(model_state, save_dir, method_name, **kwargs))


def run_async_load(save_dir, method_name):
    """wrapper to run async load in sync context"""
    return asyncio.run(load_with_tensorstore_async(save_dir, method_name))

print("async tensorstore functions for t5x loaded")

async tensorstore functions for t5x loaded


In [10]:
# tensorstore save - all parameters
tensorstore_save_dir = "saved_models/openllama_3b_tensorstore/"
os.makedirs(tensorstore_save_dir, exist_ok=True)

print("\n=== phase 2: tensorstore saving ===")
start_time = time.time()

# get all model parameters
model_state = model.state_dict()
print(f"processing {len(model_state)} parameters...")

# dynamic chunking constants (same as t5x)
_DESIRED_CHUNK_SIZE_BYTES = 64 * 1024 * 1024  # 64 mebibytes per chunk

saved_count = 0
for param_name, param_tensor in model_state.items():
    try:
        # convert pytorch tensor to numpy array for tensorstore
        # detach: remove from computation graph (no gradients)
        # cpu: move tensor from gpu to cpu memory
        # half: convert to float16 (2 bytes per value, 50% size reduction)
        # numpy: convert to numpy array (tensorstore uses numpy format)
        param_np = param_tensor.detach().cpu().half().numpy()
        
        # create filesystem-safe filename from parameter name
        # replaces dots with underscores (e.g., "model.layers.0.weight" -> "model_layers_0_weight")
        # replaces slashes with underscores (for any path-like names)
        safe_name = param_name.replace('.', '_').replace('/', '_')
        
        # define tensorstore specification for this parameter
        # spec describes storage format, location, and chunking strategy
        
        # calculate dynamic chunk shape for ~64mib chunks
        target_elements = _DESIRED_CHUNK_SIZE_BYTES // param_np.dtype.itemsize
        chunk_shape = calculate_chunk_shape(list(param_np.shape), target_elements)
        
        spec = {
            'driver': 'zarr',  # zarr: chunked, compressed array storage format
            'kvstore': {
                'driver': 'file',
                'path': f"{tensorstore_save_dir}{safe_name}.zarr"
            },
            'metadata': {
                'shape': list(param_np.shape),
                'dtype': '<f2',  # <f2 = little-endian float16 (2 bytes)
                'chunks': chunk_shape  # dynamic chunking based on tensor size
            }
        }
        
        # open tensorstore with spec and create new zarr array
        store = ts.open(spec, create=True, delete_existing=True).result()
        store.write(param_np).result()
        # write numpy array to tensorstore (synchronous operation)
        saved_count += 1
        
    except Exception as e:
        print(f"error saving {param_name}: {e}")
        continue

# save metadata
metadata = {
    'param_names': list(model_state.keys()),
    'total_params': len(model_state)
}
with open(f"{tensorstore_save_dir}metadata.json", 'w') as f:
    json.dump(metadata, f)

tensorstore_save_time = time.time() - start_time

# calculate total size
tensorstore_size = 0
for root, dirs, files in os.walk(tensorstore_save_dir):
    for file in files:
        tensorstore_size += os.path.getsize(os.path.join(root, file))
tensorstore_file_size = tensorstore_size / (1024**3)

print(f"tensorstore save completed in {tensorstore_save_time*1000:.1f} ms")
print(f"saved {saved_count} parameters")
print(f"total size: {tensorstore_file_size:.2f} gb")

del model_state
gc.collect()


=== phase 2: tensorstore saving ===
processing 237 parameters...
tensorstore save completed in 22916.8 ms
saved 237 parameters
total size: 6.38 gb


32

In [11]:
# tensorstore load
print("\n=== phase 2: tensorstore loading ===")
start_time = time.time()

# load metadata
with open(f"{tensorstore_save_dir}metadata.json", 'r') as f:
    metadata = json.load(f)

loaded_state = {}
loaded_count = 0

for param_name in metadata['param_names']:
    # iterate through all saved parameter names from metadata.json
    try:
        # reconstruct the filesystem-safe name used during saving
        safe_name = param_name.replace('.', '_').replace('/', '_')
        zarr_path = f"{tensorstore_save_dir}{safe_name}.zarr"
        # construct full path to the saved zarr file
        
        
        # check if the zarr file exists before attempting to load
        if os.path.exists(zarr_path):
            spec = {
                'driver': 'zarr',
                'kvstore': {
                    'driver': 'file',
                    'path': zarr_path
                }
            }
            
            # open the existing tensorstore for reading
            store = ts.open(spec).result()
            param_np = store.read().result()
            # read the entire array data from tensorstore
            loaded_state[param_name] = torch.from_numpy(param_np.copy())
            # convert numpy array back to pytorch tensor and store in dict
            loaded_count += 1
            
    except Exception as e:
        print(f"error loading {param_name}: {e}")
        continue

tensorstore_load_time = time.time() - start_time

print(f"tensorstore load completed in {tensorstore_load_time*1000:.1f} ms")
print(f"loaded {loaded_count} parameters successfully")

del loaded_state
gc.collect()


=== phase 2: tensorstore loading ===
tensorstore load completed in 21179.2 ms
loaded 237 parameters successfully


0

## phase 3: t5x-optimized tensorstore

In [12]:
# t5x chunking helper
# t5x chunking helper function
def calculate_chunk_shape(shape, target_elements):
    # calculate optimal chunk shape to target a specific number of elements
    # used by t5x to create ~64mib chunks for efficient i/o
    
    # ensure minimum chunk size of 1 element
    if target_elements < 1:
        target_elements = 1
    if not shape:
        return [1]
    
    chunk_shape = list(shape)
    
    # iteratively halve the largest dimension until chunk size fits target
    while np.prod(chunk_shape) > target_elements and max(chunk_shape) > 1:
        max_idx = chunk_shape.index(max(chunk_shape))
        chunk_shape[max_idx] = max(1, chunk_shape[max_idx] // 2)
    
    return chunk_shape

print("t5x helper functions loaded")

t5x helper functions loaded


In [13]:
# phase 3: t5x-optimized tensorstore saving (async i/o)
t5x_tensorstore_save_dir = "saved_models/openllama_3b_t5x_tensorstore/"
print("
=== phase 3: t5x-optimized tensorstore saving ===")

model_state = model.state_dict()
t5x_tensorstore_save_time, t5x_tensorstore_file_size = run_async_save(
    model_state,
    t5x_tensorstore_save_dir,
    "phase 3 (t5x-optimized)",
    use_compression=True,
    use_concurrency=True,
    chunk_size_mb=64
)
del model_state
gc.collect()

SyntaxError: unterminated string literal (detected at line 3) (2673580251.py, line 3)

In [ ]:
# phase 3: t5x-optimized tensorstore loading (async i/o)
t5x_tensorstore_load_time = run_async_load(
    t5x_tensorstore_save_dir,
    "phase 3 (t5x-optimized)"
)

## phase 4: additional tensorstore variations

**testing individual optimizations:**
- phase 4a: concurrency only (128 concurrent ops)
- phase 4b: large chunks only (1mb chunks)
- phase 4c: compression only (gzip)


In [ ]:
# phase 4a: tensorstore + concurrency only (dynamic 64 mib chunking)
phase4a_dir = "saved_models/phase4a_concurrency/"
print("\n=== phase 4a: concurrency only ===")

model_state = model.state_dict()
phase4a_save_time, phase4a_size = save_with_tensorstore(
    model_state,
    phase4a_dir,
    "phase 4a (concurrency)",
    use_concurrency=True,
    chunk_size_mb=64  # same as basic tensorstore
)
del model_state
gc.collect()

In [ ]:
# phase 4a: load
phase4a_load_time = load_with_tensorstore(
    phase4a_dir,
    "phase 4a (concurrency)"
)

In [ ]:
# phase 4b: tensorstore + 1 mib chunks (for comparison)
phase4b_dir = "saved_models/phase4b_chunks/"
print("\n=== phase 4b: 1 mib chunks ===")

model_state = model.state_dict()
phase4b_save_time, phase4b_size = save_with_tensorstore(
    model_state,
    phase4b_dir,
    "phase 4b (1 mib chunks)",
    chunk_size_mb=1  # smaller chunks for comparison
)
del model_state
gc.collect()

In [ ]:
# phase 4b: load
phase4b_load_time = load_with_tensorstore(
    phase4b_dir,
    "phase 4b (1 mib chunks)"
)

In [ ]:
# phase 4c: tensorstore + compression only (dynamic 64 mib chunking)
phase4c_dir = "saved_models/phase4c_compression/"
print("\n=== phase 4c: compression only ===")

model_state = model.state_dict()
phase4c_save_time, phase4c_size = save_with_tensorstore(
    model_state,
    phase4c_dir,
    "phase 4c (compression)",
    use_compression=True,
    chunk_size_mb=64  # same as basic tensorstore
)
del model_state
gc.collect()

In [ ]:
# phase 4c: load
phase4c_load_time = load_with_tensorstore(
    phase4c_dir,
    "phase 4c (compression)"
)

## performance comparison

In [ ]:
# detailed comparison table with chunking information
print("\n" + "="*120)
print("DETAILED COMPARISON TABLE - CHUNKING & STORAGE ANALYSIS")
print("="*120)

# define method details
method_details = {
    'pytorch': {
        'name': 'PyTorch',
        'chunking': 'Single file',
        'chunk_strategy': 'N/A',
        'approx_chunks': 1,
        'compression': 'None',
        'concurrency': 'Default'
    },
    'tensorstore': {
        'name': 'TensorStore (Basic)',
        'chunking': 'Dynamic 64 MiB',
        'chunk_strategy': 'Adaptive per tensor',
        'approx_chunks': 500,
        'compression': 'None',
        'concurrency': 'Default'
    },
    't5x': {
        'name': 'T5X-Optimized',
        'chunking': 'Dynamic 64 MiB',
        'chunk_strategy': 'Adaptive per tensor',
        'approx_chunks': 500,
        'compression': 'gzip level 1',
        'concurrency': '128 ops'
    },
    'ts+concurrency': {
        'name': 'TS + Concurrency',
        'chunking': 'Dynamic 64 MiB',
        'chunk_strategy': 'Adaptive per tensor',
        'approx_chunks': 500,
        'compression': 'None',
        'concurrency': '128 ops'
    },
    'ts+chunks': {
        'name': 'TS + Large Chunks',
        'chunking': '1 MiB chunks',
        'chunk_strategy': '262k elements/chunk',
        'approx_chunks': 8000,
        'compression': 'None',
        'concurrency': 'Default'
    },
    'ts+compression': {
        'name': 'TS + Compression',
        'chunking': 'Dynamic 64 MiB',
        'chunk_strategy': 'Adaptive per tensor',
        'approx_chunks': 500,
        'compression': 'gzip level 1',
        'concurrency': 'Default'
    },
        'name': 'TS + Float16',
        'chunking': 'Dynamic 64 MiB',
        'chunk_strategy': 'Adaptive per tensor',
        'approx_chunks': 500,
        'compression': 'None',
        'concurrency': 'Default'
    }
}

# print header
print(f"\n{'Method':<20} {'Save(ms)':<10} {'Load(ms)':<10} {'Size(GB)':<10} {'Chunks':<10} {'Chunking':<18} {'Compression':<15} {'Concurrency':<12}")
print("-"*120)

# print data for each method
for i, method_key in enumerate(methods):
    details = method_details[method_key]
    print(f"{details['name']:<20} {save_times[i]:<10.1f} {load_times[i]:<10.1f} {file_sizes[i]:<10.2f} "
          f"{details['approx_chunks']:<10} {details['chunking']:<18} {details['compression']:<15} {details['concurrency']:<12}")

print("\n" + "="*120)
print("CHUNK STRATEGY EXPLANATION")
print("="*120)
print("\nDynamic 64 MiB: Chunks adapt to tensor size, targeting ~64 MiB per chunk")
print("  - Small tensors (< 64 MiB): Single chunk")
print("  - Large tensors (> 64 MiB): Split into multiple ~64 MiB chunks")
print("  - Example: 4096x4096 matrix = 1 chunk (33.5 MB)")
print("  - Example: 32000x4096 embedding = 4 chunks (split into 8000x4096 each)")
print("\nDynamic 1 MiB: Fixed 262,144 elements per chunk (1 MB for float16)")
print("  - Creates more chunks but smaller individual files")
print("  - Example: 4096x4096 matrix = 64 chunks")
print("\nSingle file: Entire model in one binary file (PyTorch default)")
print("  - No chunking overhead but less flexible for distributed systems")

print("\n" + "="*120)
print("PERFORMANCE INSIGHTS")
print("="*120)
print(f"\nChunk count impact:")
print(f"  - PyTorch (1 chunk): Fastest overall ({save_times[0]:.1f}ms save)")
print(f"  - Dynamic 64 MiB (~500 chunks): Good balance ({save_times[1]:.1f}ms save)")
print(f"  - Dynamic 1 MiB (~8000 chunks): More overhead ({save_times[4]:.1f}ms save)")
print(f"\nOptimization effects:")
print(f"  - Concurrency (128 ops): {((save_times[1] - save_times[3])/save_times[1]*100):.1f}% faster save")
print(f"  - Compression (gzip): {((file_sizes[1] - file_sizes[5])/file_sizes[1]*100):.1f}% smaller size")
print(f"  - Combined (T5X): {((save_times[1] - save_times[2])/save_times[1]*100):.1f}% faster save, {((file_sizes[1] - file_sizes[2])/file_sizes[1]*100):.1f}% smaller size")

print("\n" + "="*120)
print("EXAMPLE: Large Weight Matrix (4096 x 4096 = 16.7M elements)")
print("="*120)
print("\nPyTorch:         Part of single 6.4 GB file")
print("Dynamic 64 MiB:  1 chunk (33.5 MB) - fits entirely in one chunk")
print("Dynamic 1 MiB:   64 chunks (512 KB each) - split into smaller pieces")

print("\n" + "="*120)
print("EXAMPLE: Embedding Layer (32000 x 4096 = 131M elements)")
print("="*120)
print("\nPyTorch:         Part of single 6.4 GB file")
print("Dynamic 64 MiB:  4 chunks (8000x4096 each, ~64 MB per chunk)")
print("Dynamic 1 MiB:   256 chunks (split into many small pieces)")

print("\n" + "="*120 + "\n")

In [ ]:
# create comprehensive visualization
import matplotlib.pyplot as plt
import numpy as np

# 6-way comparison - main chart
fig, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(2, 3, figsize=(20, 12))
colors = ["blue", "orange", "green", "red", "purple", "brown", "pink", "gray"]

# 1. save time comparison
ax1.bar(range(len(methods)), save_times, color=colors)
ax1.set_title("save time comparison (all methods)", fontsize=14, fontweight="bold")
ax1.set_ylabel("time (ms)", fontsize=12)
ax1.set_xticks(range(len(methods)))
ax1.set_xticklabels(methods, rotation=45, ha="right", fontsize=9)
ax1.grid(axis="y", alpha=0.3)
for i, v in enumerate(save_times):
    ax1.text(i, v*1.02, f"{v:.0f}", ha="center", fontsize=8, fontweight="bold")

# 2. load time comparison
ax2.bar(range(len(methods)), load_times, color=colors)
ax2.set_title("load time comparison (all methods)", fontsize=14, fontweight="bold")
ax2.set_ylabel("time (ms)", fontsize=12)
ax2.set_xticks(range(len(methods)))
ax2.set_xticklabels(methods, rotation=45, ha="right", fontsize=9)
ax2.grid(axis="y", alpha=0.3)
for i, v in enumerate(load_times):
    ax2.text(i, v*1.02, f"{v:.0f}", ha="center", fontsize=8, fontweight="bold")

# 3. file size comparison
ax3.bar(range(len(methods)), file_sizes, color=colors)
ax3.set_title("file size comparison (all methods)", fontsize=14, fontweight="bold")
ax3.set_ylabel("size (gb)", fontsize=12)
ax3.set_xticks(range(len(methods)))
ax3.set_xticklabels(methods, rotation=45, ha="right", fontsize=9)
ax3.grid(axis="y", alpha=0.3)
for i, v in enumerate(file_sizes):
    ax3.text(i, v*1.02, f"{v:.2f}", ha="center", fontsize=8, fontweight="bold")

# 4. save time speedup vs pytorch
pytorch_save = save_times[0]
speedup_save = [(pytorch_save / t) for t in save_times]
ax4.bar(range(len(methods)), speedup_save, color=colors)
ax4.axhline(y=1.0, color="red", linestyle="--", linewidth=2, label="pytorch baseline")
ax4.set_title("save speedup vs pytorch (higher is better)", fontsize=14, fontweight="bold")
ax4.set_ylabel("speedup factor", fontsize=12)
ax4.set_xticks(range(len(methods)))
ax4.set_xticklabels(methods, rotation=45, ha="right", fontsize=9)
ax4.grid(axis="y", alpha=0.3)
ax4.legend()
for i, v in enumerate(speedup_save):
    ax4.text(i, v*1.02, f"{v:.2f}x", ha="center", fontsize=8, fontweight="bold")

# 5. load time speedup vs pytorch
pytorch_load = load_times[0]
speedup_load = [(pytorch_load / t) for t in load_times]
ax5.bar(range(len(methods)), speedup_load, color=colors)
ax5.axhline(y=1.0, color="red", linestyle="--", linewidth=2, label="pytorch baseline")
ax5.set_title("load speedup vs pytorch (higher is better)", fontsize=14, fontweight="bold")
ax5.set_ylabel("speedup factor", fontsize=12)
ax5.set_xticks(range(len(methods)))
ax5.set_xticklabels(methods, rotation=45, ha="right", fontsize=9)
ax5.grid(axis="y", alpha=0.3)
ax5.legend()
for i, v in enumerate(speedup_load):
    ax5.text(i, v*1.02, f"{v:.2f}x", ha="center", fontsize=8, fontweight="bold")

# 6. efficiency score (combined save+load time normalized)
total_times = [s + l for s, l in zip(save_times, load_times)]
best_total = min(total_times)
efficiency = [(best_total / t) * 100 for t in total_times]
ax6.bar(range(len(methods)), efficiency, color=colors)
ax6.set_title("overall efficiency score (higher is better)", fontsize=14, fontweight="bold")
ax6.set_ylabel("efficiency %", fontsize=12)
ax6.set_xticks(range(len(methods)))
ax6.set_xticklabels(methods, rotation=45, ha="right", fontsize=9)
ax6.grid(axis="y", alpha=0.3)
for i, v in enumerate(efficiency):
    ax6.text(i, v*1.02, f"{v:.0f}%", ha="center", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.savefig("saved_models/6way_comprehensive_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✓ comprehensive analysis chart saved to: saved_models/6way_comprehensive_analysis.png")

# create individual phase comparison charts
fig2, axes = plt.subplots(2, 2, figsize=(16, 12))

# tensorstore variants comparison (phases 2, 3, 4a-4e)
ts_methods = methods[1:]  # exclude pytorch
ts_save = save_times[1:]
ts_load = load_times[1:]
ts_sizes = file_sizes[1:]
ts_colors = colors[1:]

axes[0, 0].bar(range(len(ts_methods)), ts_save, color=ts_colors)
axes[0, 0].set_title("tensorstore variants - save time", fontsize=14, fontweight="bold")
axes[0, 0].set_ylabel("time (ms)", fontsize=12)
axes[0, 0].set_xticks(range(len(ts_methods)))
axes[0, 0].set_xticklabels(ts_methods, rotation=45, ha="right", fontsize=9)
axes[0, 0].grid(axis="y", alpha=0.3)
for i, v in enumerate(ts_save):
    axes[0, 0].text(i, v*1.02, f"{v:.0f}", ha="center", fontsize=8)

axes[0, 1].bar(range(len(ts_methods)), ts_load, color=ts_colors)
axes[0, 1].set_title("tensorstore variants - load time", fontsize=14, fontweight="bold")
axes[0, 1].set_ylabel("time (ms)", fontsize=12)
axes[0, 1].set_xticks(range(len(ts_methods)))
axes[0, 1].set_xticklabels(ts_methods, rotation=45, ha="right", fontsize=9)
axes[0, 1].grid(axis="y", alpha=0.3)
for i, v in enumerate(ts_load):
    axes[0, 1].text(i, v*1.02, f"{v:.0f}", ha="center", fontsize=8)

# phase 4 optimizations comparison
phase4_methods = ["basic", "concurrency", "chunks", "compression"]
phase4_save = [save_times[1], save_times[3], save_times[4], save_times[5]]
phase4_load = [load_times[1], load_times[3], load_times[4], load_times[5]]
phase4_colors = [colors[1], colors[3], colors[4], colors[5]]

axes[1, 0].bar(range(len(phase4_methods)), phase4_save, color=phase4_colors)
axes[1, 0].set_title("phase 4 optimizations - save time", fontsize=14, fontweight="bold")
axes[1, 0].set_ylabel("time (ms)", fontsize=12)
axes[1, 0].set_xticks(range(len(phase4_methods)))
axes[1, 0].set_xticklabels(phase4_methods, rotation=45, ha="right", fontsize=9)
axes[1, 0].grid(axis="y", alpha=0.3)
for i, v in enumerate(phase4_save):
    axes[1, 0].text(i, v*1.02, f"{v:.0f}", ha="center", fontsize=8)

axes[1, 1].bar(range(len(phase4_methods)), phase4_load, color=phase4_colors)
axes[1, 1].set_title("phase 4 optimizations - load time", fontsize=14, fontweight="bold")
axes[1, 1].set_ylabel("time (ms)", fontsize=12)
axes[1, 1].set_xticks(range(len(phase4_methods)))
axes[1, 1].set_xticklabels(phase4_methods, rotation=45, ha="right", fontsize=9)
axes[1, 1].grid(axis="y", alpha=0.3)
for i, v in enumerate(phase4_load):
    axes[1, 1].text(i, v*1.02, f"{v:.0f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("saved_models/tensorstore_variants_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ tensorstore variants chart saved to: saved_models/tensorstore_variants_comparison.png")

# create optimization impact chart
fig3, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(16, 6))

# improvement percentages vs basic tensorstore
baseline_save = save_times[1]  # basic tensorstore
baseline_load = load_times[1]
improvements_save = [((baseline_save - s) / baseline_save * 100) for s in save_times[2:]]
improvements_load = [((baseline_load - l) / baseline_load * 100) for l in load_times[2:]]
opt_methods = methods[2:]
opt_colors = colors[2:]

x_pos = np.arange(len(opt_methods))
width = 0.35
ax_a.bar(x_pos - width/2, improvements_save, width, label="save time", color="lightcoral", alpha=0.8)
ax_a.bar(x_pos + width/2, improvements_load, width, label="load time", color="lightblue", alpha=0.8)
ax_a.axhline(y=0, color="red", linestyle="--", linewidth=2, alpha=0.5)
ax_a.set_title("improvement vs basic tensorstore", fontsize=14, fontweight="bold")
ax_a.set_ylabel("improvement %", fontsize=12)
ax_a.set_xticks(x_pos)
ax_a.set_xticklabels(opt_methods, rotation=45, ha="right", fontsize=9)
ax_a.legend()
ax_a.grid(axis="y", alpha=0.3)
for i, (s, l) in enumerate(zip(improvements_save, improvements_load)):
    ax_a.text(i - width/2, s + (2 if s > 0 else -2), f"{s:+.0f}%", ha="center", fontsize=8)
    ax_a.text(i + width/2, l + (2 if l > 0 else -2), f"{l:+.0f}%", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("saved_models/optimization_impact_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ optimization impact chart saved to: saved_models/optimization_impact_analysis.png")

print("\n📊 all visualization charts generated successfully!")